# Phase 2 — Two-stage upscaler (x4-upscaler + ControlNet Tile img2img)

Runs `UpscalerPipeline.upscale_two_stage` across the 60-image test set at 4× / 5× / 10×. The key experiment is how much stage-B denoise improves fine detail before the 10× cliff bites.

**Outputs in `outputs/phase2/`:**
- `sweep/` — per-image parameter sweep (4 curated images × 3 ratios × 3 denoise settings)
- `per_image/` — per-image grids for all 60 images × 3 ratios at the recommended denoise
- `fidelity_cliff.png` — one explicitly annotated image showing the 5×→10× cliff

**Runtime:** ~1 hour on the local 5070. All upscales cached to `outputs/phase2/upscales/` so re-runs are fast.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image
from tqdm.auto import tqdm

from upscaler import pipeline, testset

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PHASE2_DIR = REPO_ROOT / "outputs" / "phase2"
CACHE_DIR = PHASE2_DIR / "upscales"
SWEEP_DIR = PHASE2_DIR / "sweep"
PERIMG_DIR = PHASE2_DIR / "per_image"
for d in (CACHE_DIR, SWEEP_DIR, PERIMG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"torch {torch.__version__}, cuda_available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")
print(f"outputs -> {PHASE2_DIR}")

In [ ]:
ts = testset.load(REPO_ROOT / "data" / "test_images")
LR_SIZES = (100, 200, 250)  # 10x, 5x, 4x
TARGET = 1000
PROMPT = "a high-resolution photograph, sharp detail, professional quality"
print(f"loaded {len(ts)} images")

In [ ]:
up = pipeline.UpscalerPipeline()
up.load()
up.load_stage_b()
print("pipelines loaded")

In [ ]:
def cached_two_stage(img_stem: str, lr_size: int, denoise: float, steps: int = 20) -> Path:
    key = f"{img_stem}_{lr_size}_d{round(denoise * 100):03d}_s{steps}"
    out_path = CACHE_DIR / f"{key}.jpg"
    if out_path.is_file():
        return out_path
    lr = Image.open(ts.root / f"{img_stem}_{lr_size}.jpg").convert("RGB")
    out = up.upscale_two_stage(
        lr,
        target_size=TARGET,
        denoise=denoise,
        steps=steps,
        cn_weight=1.0,
        prompt=PROMPT,
    )
    out.save(out_path, quality=92)
    return out_path

## Parameter sweep on curated subset

Four images covering distinct regimes. Each rendered at three ratios × three denoise strengths so the effect of stage-B influence is legible.

In [ ]:
CURATED = [
    "California_Wildflowers",  # traditional / landscape
    "Athens_Street",  # traditional / cityscape
    "Duomo_fine-arch",  # hard / fine_architecture
    "Forest_hf-texture",  # hard / hf_texture
]
SWEEP_DENOISE = (0.20, 0.35, 0.55)

sweep_work = [(stem, lr, d) for stem in CURATED for lr in LR_SIZES for d in SWEEP_DENOISE]

for stem, lr, d in tqdm(sweep_work, desc="sweep"):
    cached_two_stage(stem, lr, d)

print(f"{len(sweep_work)} sweep upscales cached")

In [ ]:
def render_sweep_grid(img_stem: str, out_path: Path, crop_size: int = 400) -> None:
    """Rows: LR ratio (4x / 5x / 10x). Cols: denoise 0.20 / 0.35 / 0.55 + HR reference.

    Center-crop so pixel-level differences pop at a legible size.
    """
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    cx = (TARGET - crop_size) // 2
    crop_box = (cx, cx, cx + crop_size, cx + crop_size)
    ratios = [(250, "4x"), (200, "5x"), (100, "10x")]
    hr = Image.open(ts.root / f"{img_stem}.jpg").convert("RGB").crop(crop_box)

    fig.suptitle(f"{img_stem}  (center {crop_size}x{crop_size} crop)", fontsize=14)
    for r, (lr_size, ratio_name) in enumerate(ratios):
        axes[r, 0].imshow(hr)
        axes[r, 0].axis("off")
        if r == 0:
            axes[r, 0].set_title("HR reference", fontsize=11)
        axes[r, 0].text(
            -0.08,
            0.5,
            ratio_name,
            rotation=90,
            ha="center",
            va="center",
            transform=axes[r, 0].transAxes,
            fontsize=13,
            fontweight="bold",
        )
        for c, d in enumerate(SWEEP_DENOISE, start=1):
            p = cached_two_stage(img_stem, lr_size, d)
            axes[r, c].imshow(Image.open(p).convert("RGB").crop(crop_box))
            axes[r, c].axis("off")
            if r == 0:
                axes[r, c].set_title(f"denoise {d}", fontsize=11)

    fig.tight_layout(rect=[0.02, 0, 1, 0.97])
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close(fig)


for stem in CURATED:
    render_sweep_grid(stem, SWEEP_DIR / f"{stem}.png")
print(f"{len(list(SWEEP_DIR.glob('*.png')))} sweep grids in {SWEEP_DIR}")

## Main run: all 60 images × 3 ratios at recommended denoise (0.30)

Per-image grids showing stage-A-only (4x native) vs two-stage at each ratio. The visual headline of Phase 2.

In [ ]:
MAIN_DENOISE = 0.30

main_work = [(img.name, lr) for img in ts for lr in LR_SIZES]
for name, lr in tqdm(main_work, desc="main run"):
    stem = Path(name).stem
    cached_two_stage(stem, lr, MAIN_DENOISE)
print(f"{len(main_work)} main upscales cached")

In [ ]:
def render_per_image_grid(img: testset.TestImage, out_path: Path, crop_size: int = 400) -> None:
    """One row per ratio, columns = LR (upscaled-for-display) | two-stage | HR reference."""
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    cx = (TARGET - crop_size) // 2
    crop_box = (cx, cx, cx + crop_size, cx + crop_size)
    ratios = [(250, "4x (250→1000)"), (200, "5x (200→1000)"), (100, "10x (100→1000)")]
    hr = Image.open(img.hr_path).convert("RGB").crop(crop_box)
    stem = Path(img.name).stem

    fig.suptitle(f"{stem}  (center {crop_size}x{crop_size} crop)", fontsize=14)
    for r, (lr_size, label) in enumerate(ratios):
        lr_img = Image.open(img.lr_path(lr_size)).convert("RGB")
        lr_display = lr_img.resize((TARGET, TARGET), Image.Resampling.NEAREST).crop(crop_box)
        axes[r, 0].imshow(lr_display)
        axes[r, 0].axis("off")
        axes[r, 0].set_title(f"LR input, {label}" if r == 0 else label, fontsize=11)

        two_stage = (
            Image.open(cached_two_stage(stem, lr_size, MAIN_DENOISE)).convert("RGB").crop(crop_box)
        )
        axes[r, 1].imshow(two_stage)
        axes[r, 1].axis("off")
        if r == 0:
            axes[r, 1].set_title(f"two-stage (denoise {MAIN_DENOISE})", fontsize=11)

        axes[r, 2].imshow(hr)
        axes[r, 2].axis("off")
        if r == 0:
            axes[r, 2].set_title("HR reference", fontsize=11)

    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close(fig)


for img in tqdm(ts.images, desc="per-image grids"):
    render_per_image_grid(img, PERIMG_DIR / f"{Path(img.name).stem}.png")
print(f"{len(list(PERIMG_DIR.glob('*.png')))} per-image grids in {PERIMG_DIR}")

## Fidelity cliff annotation

One image, rendered explicitly to show how 10× (100→1000) breaks down where 5× still holds. Stage B invents reasonable texture but can't recover scene structure that wasn't in the LR input.

In [ ]:
from IPython.display import Image as IPImage

CLIFF_IMG = "Duomo_fine-arch"  # fine architecture: structure is hardest to hallucinate at 10x
CLIFF_DENOISE = 0.35  # stage-B a bit heavier so the cliff is obvious

for lr in LR_SIZES:
    cached_two_stage(CLIFF_IMG, lr, CLIFF_DENOISE)

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
crop_size = 500
cx = (TARGET - crop_size) // 2
crop_box = (cx, cx, cx + crop_size, cx + crop_size)
hr = Image.open(ts.root / f"{CLIFF_IMG}.jpg").convert("RGB").crop(crop_box)

axes[0].imshow(hr)
axes[0].set_title("HR reference (1000×1000)", fontsize=12)
axes[0].axis("off")

labels = ["4× (250→1000)", "5× (200→1000)", "10× (100→1000)"]
for i, (lr_size, label) in enumerate(
    zip(LR_SIZES[::-1], labels, strict=True)
):  # 250, 200, 100 order
    p = cached_two_stage(CLIFF_IMG, lr_size, CLIFF_DENOISE)
    axes[i + 1].imshow(Image.open(p).convert("RGB").crop(crop_box))
    axes[i + 1].set_title(label, fontsize=12)
    axes[i + 1].axis("off")

# Annotate the cliff: arrow over the 10x pane, text call-out.
axes[3].annotate(
    "fidelity cliff:\nornate window mullions\ndissolve into plausible\nbut invented texture",
    xy=(0.5, 0.5),
    xycoords="axes fraction",
    xytext=(1.05, 0.5),
    textcoords="axes fraction",
    ha="left",
    va="center",
    fontsize=11,
    color="darkred",
    arrowprops=dict(arrowstyle="->", color="darkred", lw=1.5),
)
fig.suptitle(f"{CLIFF_IMG} — two-stage at denoise {CLIFF_DENOISE}", fontsize=14)
fig.tight_layout(rect=[0, 0, 0.92, 0.95])
fig.savefig(PHASE2_DIR / "fidelity_cliff.png", dpi=130, bbox_inches="tight")
plt.close(fig)

IPImage(filename=str(PHASE2_DIR / "fidelity_cliff.png"))

In [ ]:
up.close()
print("pipelines released")